In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, Literal
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
llm = ChatGoogleGenerativeAI()

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]  # BaseMessage contain all the messages ->HumanMessage, SystemMessage etc
    # prefer add_messages when using BaseMessages

In [ ]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = llm.invoke(messages)
    return {'messages': [response]}

In [ ]:
checkpointer = MemorySaver() # it saves the previous messages(RAM or database) as chat ends 
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
thread_id = '1'# many users can chat so each user have unique id where his/her chat will be stored

while True:

    user_message = input('type here')
    print('user:', user_message)
    if user_message.strip().lower in ['exit', 'quit', 'bye']:
        print('thank you')
        break

    config = {'configurable': {'thread_id': thread_id}}
    response = chatbot.invoke({'messages': [HumanMessage(content= user_message)]}, config=config)
    print('AI:', response['messages'][-1].content)